# PreProcess Stage 3 
## Zarr Writer

## Prerequisites
Ensure that the previous stage has has been run and the following two files are available: 


- `intermediate_grouper.json` — the grouped image data
- `pipeline_stats.json` — the timing and summary information

## Select data paths
Define the config.yaml path, the name of the intermediate json and the name of the output stats json. If unsure, do not change the defaults.
If Stage 1 and 2 were ran with default settings, these will already be correct

In [ ]:
!cd ..
CONFIG_PATH   = "pre_process/config.yaml"
IN_DATA       = "intermediate_filtered.json"
OUT_MANIFESTS = "intermediate_zarr_manifests.json"
STATS_FILE    = "pipeline_stats.json"

## Load filtered dataset
The subsequent Cell of code will load the filtered dataset (created from stage 2)

In [ ]:
import json
from pathlib import Path

from pre_process._pre_process_utils.pipeline_utils import load_config, deserialise_buckets
from pre_process.zarr_writer import ZarrWriterConfig

config = load_config(CONFIG_PATH)
writer_cfg = ZarrWriterConfig.from_dict(config)

filtered_buckets = deserialise_buckets(json.loads(Path(IN_DATA).read_text()))
pipeline_stats = json.loads(Path(STATS_FILE).read_text())

total = sum(len(v) for v in filtered_buckets.values())
print(f"Loaded {total} images across {len(filtered_buckets)} buckets")

## Write Image arrays
Each group of images (grouped by resolution in stage 1, then filtered in stage 2) is now stored as a compressed array file on the local storage (Zarr format)

In [ ]:
import time
from pre_process.zarr_writer import ZarrWriter

t0 = time.perf_counter()
writer = ZarrWriter(writer_cfg)
zarr_manifests = writer.write_all(filtered_buckets)
dt = time.perf_counter() - t0

print(f"Zarr writing complete in {dt:.1f}s")
print(f"Arrays written: {len(zarr_manifests)}")

## Export Results
A record (manifest) is now written of every array that was created and updates the pipeline stats

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pre_process._pre_process_utils.pipeline_utils import safe_write_json

manifest_dicts = [m.to_dict() for m in zarr_manifests]
pipeline_stats["timing"]["zarr_write_s"] = round(dt, 2)
pipeline_stats["zarr_manifests"] = manifest_dicts

with ThreadPoolExecutor(max_workers=2) as executor:
    f1 = executor.submit(safe_write_json, manifest_dicts, OUT_MANIFESTS)
    f2 = executor.submit(safe_write_json, pipeline_stats, STATS_FILE)
    f1.result()
    f2.result()

print(f"Manifests saved to: {OUT_MANIFESTS}")
print(f"Stats updated in:   {STATS_FILE}")

## (Opt) Sanity check Manifests
Run the subsequent cell to check the manifest of each array that was written (not recommended for production usage)

In [ ]:
print(f"{'Bucket':<20} {'Before':>8} {'After':>8} {'Rejected':>10}")
print("-" * 50)
for m in manifest_dicts:
    print(json.dumps(m, indent=2))
    print()